# 04 — Comparison: YOLOv8n vs YOLO11n vs YOLO26n (Aksara Jawa Classification)

Notebook ini menguji apakah **upgrade arsitektur YOLO** (terutama penambahan blok attention C2PSA) memberikan peningkatan akurasi yang signifikan pada klasifikasi aksara Jawa.

## 3-tier comparison

| Era | Model | Tahun | Attention block? |
|---|---|---|---|
| Pre-attention | **YOLOv8n-cls** | 2023 | ❌ tidak ada |
| Attention era | **YOLO11n-cls** | 2024 | ✅ C2PSA |
| Latest | **YOLO26n-cls** | 2025 | ✅ C2PSA |

## Hipotesis
- **H0**: Tidak ada perbedaan akurasi yang signifikan antara ketiga model (ANOVA p ≥ 0.05)
- **H1**: Setidaknya satu pasang model berbeda secara signifikan (p < 0.05)

Jika H0 diterima → bukti bahwa **attention block (C2PSA) tidak diperlukan** untuk klasifikasi aksara Jawa.

## Protokol training (sama untuk semua model)
- 50 epochs, patience 5
- imgsz 128, batch 64, workers 4
- device GPU 0 (RTX 5080)
- 5 seeds: `[42, 123, 7, 2024, 999]`

## Struktur notebook
1. Single-run comparison (YOLO11 vs YOLO26, seed=42) — quick look
2. **Multi-seed test (3 models × 5 seeds = 15 runs)** — robust statistical analysis
3. ANOVA + pairwise paired t-tests
4. Visualisasi (bar chart + per-seed plot)

In [1]:
import time
from pathlib import Path
import torch
from ultralytics import YOLO

ROOT = Path('D:/MyPaper/AksaraJawa')
DATA = ROOT / 'dataset' / 'DatasetPhiard'
RUNS = ROOT / 'runs'

print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))

CUDA: True | NVIDIA GeForce RTX 5080 Laptop GPU


## Train YOLO11n-cls

## Train YOLOv8n-cls (pre-attention era baseline)

In [2]:
model8 = YOLO('yolov8n-cls.pt')
t0 = time.time()
res8 = model8.train(
    data=str(DATA), epochs=50, imgsz=128, batch=64, device=0, workers=4,
    project=str(RUNS), name='clean_yolov8n', exist_ok=True,
    patience=5, plots=True, seed=42, verbose=False,
)
t8 = time.time() - t0
print(f'\n[YOLOv8n] training time: {t8:.1f}s')

Ultralytics 8.4.46  Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080 Laptop GPU, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Skripsi\AksaraJawa\DatasetPhiard, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=128, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=clean_yolov8n, nbs=64, nms=False, opset=None, optimize=False, optimizer

In [3]:
model11 = YOLO('yolo11n-cls.pt')
t0 = time.time()
res11 = model11.train(
    data=str(DATA), epochs=50, imgsz=128, batch=64, device=0, workers=4,
    project=str(RUNS), name='clean_yolo11n', exist_ok=True,
    patience=5, plots=True, seed=42, verbose=False,
)
t11 = time.time() - t0
print(f'\n[YOLO11n] training time: {t11:.1f}s')

Ultralytics 8.4.46  Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080 Laptop GPU, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Skripsi\AksaraJawa\DatasetPhiard, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=128, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=clean_yolo11n, nbs=64, nms=False, opset=None, optimize=False, optimizer

## Train YOLO26n-cls

In [4]:
model26 = YOLO(str(ROOT / 'weights' / 'yolo26n-cls.pt'))
t0 = time.time()
res26 = model26.train(
    data=str(DATA), epochs=50, imgsz=128, batch=64, device=0, workers=4,
    project=str(RUNS), name='clean_yolo26n_compare', exist_ok=True,
    patience=5, plots=True, seed=42, verbose=False,
)
t26 = time.time() - t0
print(f'\n[YOLO26n] training time: {t26:.1f}s')

Ultralytics 8.4.46  Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080 Laptop GPU, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Skripsi\AksaraJawa\DatasetPhiard, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=128, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\Skripsi\AksaraJawa\weights\yolo26n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=clean_yolo26n_compare, nbs=64, nms=False,

## Validation metrics

In [5]:
val8  = model8.val(verbose=False)
val11 = model11.val(verbose=False)
val26 = model26.val(verbose=False)

# Inference speed test
import random
from PIL import Image
pred_files = list((ROOT / 'dataset' / 'Dataset' / 'prediction' / 'prediction').glob('*.png'))[:50]

def bench(model, files, warmup=5):
    for f in files[:warmup]: model.predict(str(f), imgsz=128, verbose=False)
    t0 = time.time()
    for f in files: model.predict(str(f), imgsz=128, verbose=False)
    return (time.time() - t0) / len(files) * 1000

ms8  = bench(model8, pred_files)
ms11 = bench(model11, pred_files)
ms26 = bench(model26, pred_files)

n_params_8  = sum(p.numel() for p in model8.model.parameters())
n_params_11 = sum(p.numel() for p in model11.model.parameters())
n_params_26 = sum(p.numel() for p in model26.model.parameters())

Ultralytics 8.4.46  Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080 Laptop GPU, 16303MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,460,500 parameters, 0 gradients, 3.3 GFLOPs
train: D:\Skripsi\AksaraJawa\DatasetPhiard\train... found 2076 images in 20 classes  
val: D:\Skripsi\AksaraJawa\DatasetPhiard\val... found 558 images in 20 classes  
test: None...
val: Fast image access  (ping: 0.00.0 ms, read: 261.8143.2 MB/s, size: 13.2 KB)
val: Scanning D:\Skripsi\AksaraJawa\DatasetPhiard\val... 558 images, 0 corrupt: 100% ━━━━━━━━━━━━ 558/558  0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 35/35 106.4it/s 0.3s.2s
                   all      0.995          1
Speed: 0.1ms preprocess, 0.4ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to D:\Skripsi\AksaraJawa\notebooks\runs\classify\val-82
Ultralytics 8.4.46  Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080 Laptop GPU, 16303MiB)
YOLO11n-cls summary (fused): 47 layers, 

## Tabel perbandingan

In [6]:
import pandas as pd

df = pd.DataFrame({
    'Metric': ['Top-1 acc', 'Top-5 acc', 'Total params', 'Train time (s)', 'Inference (ms/img)'],
    'YOLOv8n-cls':  [f'{val8.top1:.4f}',  f'{val8.top5:.4f}',
                     f'{n_params_8:,}',  f'{t8:.1f}',  f'{ms8:.2f}'],
    'YOLO11n-cls': [f'{val11.top1:.4f}', f'{val11.top5:.4f}',
                    f'{n_params_11:,}', f'{t11:.1f}', f'{ms11:.2f}'],
    'YOLO26n-cls': [f'{val26.top1:.4f}', f'{val26.top5:.4f}',
                    f'{n_params_26:,}', f'{t26:.1f}', f'{ms26:.2f}'],
})
print(df.to_string(index=False))

            Metric YOLOv8n-cls YOLO11n-cls YOLO26n-cls
         Top-1 acc      0.9946      0.9857      0.9803
         Top-5 acc      1.0000      1.0000      1.0000
      Total params   1,460,500   1,551,644   1,551,644
    Train time (s)        37.3        49.0        42.4
Inference (ms/img)       15.40       15.33       14.95


## Plot kurva training (loss + acc)

In [7]:
import matplotlib.pyplot as plt
import pandas as pd

csv8  = RUNS / 'aksara_yolov8n' / 'results.csv'
csv11 = RUNS / 'aksara_yolo11n' / 'results.csv'
csv26 = RUNS / 'aksara_yolo26n_compare' / 'results.csv'

d8  = pd.read_csv(csv8);  d8.columns  = d8.columns.str.strip()
d11 = pd.read_csv(csv11); d11.columns = d11.columns.str.strip()
d26 = pd.read_csv(csv26); d26.columns = d26.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for d, label, color, mk in [(d8, 'YOLOv8n', 'mediumseagreen', '^'),
                             (d11, 'YOLO11n', 'steelblue', 'o'),
                             (d26, 'YOLO26n', 'salmon', 's')]:
    axes[0].plot(d['epoch'], d['train/loss'], mk + '-', label=label, color=color)
    axes[1].plot(d['epoch'], d['val/loss'], mk + '-', label=label, color=color)
    axes[2].plot(d['epoch'], d['metrics/accuracy_top1'], mk + '-', label=label, color=color)

axes[0].set_title('Train loss');     axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
axes[1].set_title('Val loss');       axes[1].set_xlabel('epoch'); axes[1].set_ylabel('loss')
axes[2].set_title('Val Top-1 acc');  axes[2].set_xlabel('epoch'); axes[2].set_ylabel('top-1 acc')
for a in axes: a.legend(); a.grid(alpha=0.3)

plt.suptitle('YOLOv8n vs YOLO11n vs YOLO26n — Aksara Jawa Classification', fontsize=13)
plt.tight_layout()
plt.savefig(ROOT / 'runs' / 'compare_3models_clean.png', dpi=120, bbox_inches='tight')
plt.show()

<Figure size 1600x450 with 3 Axes>

## Confusion matrix dua model side-by-side

In [8]:
from PIL import Image
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, run, label in [(axes[0], 'aksara_yolov8n',         'YOLOv8n'),
                        (axes[1], 'aksara_yolo11n',         'YOLO11n'),
                        (axes[2], 'aksara_yolo26n_compare', 'YOLO26n')]:
    cm = RUNS / run / 'confusion_matrix_normalized.png'
    if cm.exists():
        ax.imshow(Image.open(cm))
        ax.set_title(label, fontsize=12)
    ax.axis('off')
plt.suptitle('Confusion Matrix Comparison (3 Models)', fontsize=13)
plt.tight_layout(); plt.show()

<Figure size 1800x600 with 3 Axes>

## Kesimpulan

Cek tabel & plot di atas. Tipikal hasil yang diharapkan:
- **Akurasi**: beda <1%, kemungkinan besar tied (random seed effect)
- **Param**: hampir identik (~1.5M)
- **Train time**: sebanding
- **Inference**: YOLO26 sedikit lebih cepat (efek arsitektur engineering tweaks)

**Implikasi buat paper**:
> Untuk klasifikasi aksara Jawa, perbedaan arsitektur YOLO11 vs YOLO26 tidak menghasilkan peningkatan akurasi yang signifikan, mengkonfirmasi bahwa task klasifikasi karakter sederhana tidak benefit besar dari upgrade arsitektur YOLO terbaru. Pemilihan model bisa berbasis pertimbangan praktis (latency, dukungan komunitas, kemudahan deployment).

---

# 🔬 Multi-Seed Robustness Test (5 seeds)

Train kedua model masing-masing dengan **5 seed berbeda** untuk dapet **mean ± std** yang valid secara statistik. Total: 10 training run, perkiraan ~12-15 menit di RTX 5080.

Hasil ini lebih robust untuk klaim ilmiah di paper.

In [9]:
import gc
SEEDS = [42, 123, 7, 2024, 999]
RUNS_MS = ROOT / 'runs' / 'multiseed_clean'
RUNS_MS.mkdir(parents=True, exist_ok=True)

# 3 model comparison: era pre-attention vs attention era vs latest
MODELS = [
    ('yolov8n', 'yolov8n-cls.pt'),                              # pre-attention era (2023)
    ('yolo11n', 'yolo11n-cls.pt'),                              # attention era (2024, C2PSA)
    ('yolo26n', str(ROOT / 'weights' / 'yolo26n-cls.pt')),      # latest (2025, C2PSA)
]

results_ms = []  # list of dict: model, seed, top1, top5, time_s

def train_one(weights, model_tag, seed):
    m = YOLO(weights)
    t0 = time.time()
    m.train(
        data=str(DATA), epochs=50, imgsz=128, batch=64, device=0, workers=4,
        project=str(RUNS_MS), name=f'{model_tag}_seed{seed}', exist_ok=True,
        patience=5, plots=False, seed=seed, verbose=False,
    )
    dt = time.time() - t0
    val = m.val(verbose=False)
    out = {'model': model_tag, 'seed': seed,
           'top1': float(val.top1), 'top5': float(val.top5),
           'time_s': dt}
    del m, val
    gc.collect()
    torch.cuda.empty_cache()
    return out

print(f'Mulai multi-seed training: {len(MODELS)} models x {len(SEEDS)} seeds = {len(MODELS)*len(SEEDS)} runs\n')
for seed in SEEDS:
    print(f'=== seed={seed} ===')
    for tag, weights in MODELS:
        r = train_one(weights, tag, seed)
        print(f'  {tag}: top1={r["top1"]:.4f}  time={r["time_s"]:.1f}s')
        results_ms.append(r)

print('\nSelesai semua run.')

Mulai multi-seed training: 3 models x 5 seeds = 15 runs

=== seed=42 ===
Ultralytics 8.4.46  Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080 Laptop GPU, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Skripsi\AksaraJawa\DatasetPhiard, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=128, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, nam

## Statistik mean ± std + uji signifikansi (paired t-test)

In [10]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations

dfm = pd.DataFrame(results_ms)
print('Raw results per seed:')
print(dfm.pivot(index='seed', columns='model', values='top1').round(4))
print()

agg = dfm.groupby('model').agg(
    top1_mean=('top1', 'mean'), top1_std=('top1', 'std'),
    top5_mean=('top5', 'mean'), top5_std=('top5', 'std'),
    time_mean=('time_s', 'mean'),
).round(4)
print('Agregat per model:')
print(agg)

model_tags = [t for t, _ in MODELS]
top1_data = {t: dfm[dfm.model == t]['top1'].values for t in model_tags}

# ANOVA: ada perbedaan signifikan antara minimal salah satu pair?
f_stat, p_anova = stats.f_oneway(*[top1_data[t] for t in model_tags])
print(f'\nOne-way ANOVA across {len(model_tags)} models:')
print(f'  F-statistic = {f_stat:.3f}')
print(f'  p-value     = {p_anova:.4f}   {"(SIGNIFIKAN p<0.05)" if p_anova < 0.05 else "(tidak signifikan, p>=0.05)"}')

# Pairwise paired t-test (semua kombinasi)
print(f'\nPairwise paired t-tests:')
pairwise = []
for a, b in combinations(model_tags, 2):
    t_stat, p = stats.ttest_rel(top1_data[a], top1_data[b])
    diff = top1_data[a].mean() - top1_data[b].mean()
    sig = '*' if p < 0.05 else ''
    pairwise.append({'pair': f'{a} vs {b}', 'mean_diff_pct': diff*100,
                     't': t_stat, 'p': p, 'sig': sig})
    print(f'  {a:8s} vs {b:8s}:  diff={diff*100:+.2f}%   t={t_stat:+.3f}   p={p:.4f}  {sig}')

print('\n* = signifikan (p<0.05)')

Raw results per seed:
model  yolo11n  yolo26n  yolov8n
seed                            
7       0.9713   0.9928   0.9928
42      0.9857   0.9803   0.9946
123     0.9767   0.9964   0.9946
999     0.9857   0.9892   0.9928
2024    0.9928   0.9928   0.9928

Agregat per model:
         top1_mean  top1_std  top5_mean  top5_std  time_mean
model                                                       
yolo11n     0.9824    0.0084        1.0       0.0    41.1099
yolo26n     0.9903    0.0062        1.0       0.0    45.6235
yolov8n     0.9935    0.0010        1.0       0.0    37.3446

One-way ANOVA across 3 models:
  F-statistic = 4.449
  p-value     = 0.0358   (SIGNIFIKAN p<0.05)

Pairwise paired t-tests:
  yolov8n  vs yolo11n :  diff=+1.11%   t=+2.878   p=0.0451  *
  yolov8n  vs yolo26n :  diff=+0.32%   t=+1.108   p=0.3301  
  yolo11n  vs yolo26n :  diff=-0.79%   t=-1.462   p=0.2176  

* = signifikan (p<0.05)


## Plot bar chart dengan error bar + scatter per seed

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart mean ± std untuk semua model
labels = [t for t, _ in MODELS]
means = [top1_data[t].mean() for t in labels]
stds  = [top1_data[t].std() for t in labels]
palette = ['mediumseagreen', 'steelblue', 'salmon']

axes[0].bar(labels, means, yerr=stds, capsize=8, color=palette, alpha=0.85,
            edgecolor='black', linewidth=1)
for i, t in enumerate(labels):
    axes[0].scatter([i] * len(top1_data[t]), top1_data[t], color='black', zorder=10, s=40, alpha=0.7)
axes[0].set_ylabel('Top-1 accuracy')
axes[0].set_title(f'Mean ± std (n={len(SEEDS)} seeds)\nANOVA p={p_anova:.4f} '
                  f'({"signifikan" if p_anova<0.05 else "tidak signifikan"})')
axes[0].grid(axis='y', alpha=0.3)
ymin = min(means) - max(stds) * 3
ymax = max(means) + max(stds) * 3
axes[0].set_ylim(ymin, ymax)

# Per-seed plot, line per seed connecting all 3 models
for seed in sorted(SEEDS):
    vals = [dfm[(dfm.model == t) & (dfm.seed == seed)]['top1'].values[0] for t in labels]
    axes[1].plot(range(len(labels)), vals, 'o-', label=f'seed={seed}', alpha=0.7)
axes[1].set_xticks(range(len(labels))); axes[1].set_xticklabels(labels)
axes[1].set_ylabel('Top-1 accuracy')
axes[1].set_title('Per-seed comparison (3 models)')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.suptitle('YOLOv8n vs YOLO11n vs YOLO26n — Aksara Jawa (Multi-Seed)', fontsize=12)
plt.tight_layout()
plt.savefig(ROOT / 'runs' / 'multiseed_clean_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

<Figure size 1400x500 with 2 Axes>

## Save raw results untuk dokumentasi paper

In [12]:
out_csv = ROOT / 'runs' / 'multiseed_clean_results.csv'
dfm.to_csv(out_csv, index=False)
print(f'Raw results saved -> {out_csv}')

# Summary text untuk paper
lines = [f'=== Multi-Seed Comparison Summary (n={len(SEEDS)} seeds) ===',
         f'Seeds: {SEEDS}',
         '']
for t in [tag for tag, _ in MODELS]:
    lines.append(f'{t} top-1: {top1_data[t].mean()*100:.2f}% ± {top1_data[t].std()*100:.2f}%')
lines += ['',
          f'One-way ANOVA: F={f_stat:.3f}, p={p_anova:.4f}',
          f'  -> {"SIGNIFIKAN" if p_anova<0.05 else "TIDAK SIGNIFIKAN"} (alpha=0.05)',
          '',
          'Pairwise paired t-tests:']
for pw in pairwise:
    lines.append(f'  {pw["pair"]:25s}  diff={pw["mean_diff_pct"]:+.2f}%   p={pw["p"]:.4f}  {pw["sig"]}')

summary = '\n'.join(lines)
print('\n' + summary)
(ROOT / 'runs' / 'multiseed_clean_summary.txt').write_text(summary, encoding='utf-8')

Raw results saved -> D:\Skripsi\AksaraJawa\runs\multiseed_clean_results.csv

=== Multi-Seed Comparison Summary (n=5 seeds) ===
Seeds: [42, 123, 7, 2024, 999]

yolov8n top-1: 99.35% ± 0.09%
yolo11n top-1: 98.24% ± 0.76%
yolo26n top-1: 99.03% ± 0.55%

One-way ANOVA: F=4.449, p=0.0358
  -> SIGNIFIKAN (alpha=0.05)

Pairwise paired t-tests:
  yolov8n vs yolo11n         diff=+1.11%   p=0.0451  *
  yolov8n vs yolo26n         diff=+0.32%   p=0.3301  
  yolo11n vs yolo26n         diff=-0.79%   p=0.2176  


423